# 分類模型評估

對應 DLI：04b、05。

本課使用小型合成資料或 scikit-learn 內建資料，在 CPU 執行，不需下載原課程資料。依序執行所有儲存格。

[在 Google Colab 執行本課](https://colab.research.google.com/github/Samuel-ncu/AI-Stastic/blob/main/notebooks/07_classification_evaluation.ipynb) · [閱讀完整背景講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/07-classification-evaluation.md)

建議順序：先讀概念與手算 → 回答檢核 → 執行程式 → 修改練習。

## 先定用途，再設計閾值、規則和評估方式

**閾值是依任務需求設計的。** 先決定系統要採取什麼行動、有哪些限制，再思考怎麼用模型輸出。常見做法如下：

| 從哪裡著手？ | 怎麼用？ | 例子 |
|---|---|---|
| **從單筆分數定門檻** | 分數達到門檻就採取行動。門檻也可以來自現成規格。 | 瑕疵機率 42%，用 50% 門檻就放行，用 30% 就送複檢。若模型輸出的是延遲毫秒數，服務可依 12 ms 的規格決定何時警報。 |
| **從任務設判定規則** | 依最高分、類別或額外條件決定單筆要做什麼。 | DLI 04b 直接選最高分的手語字母。DLI 05a 先選最高分的 ImageNet 類別，再依該類別是否屬犬種決定開門。 |
| **用整批驗證選規則** | 試幾個候選門檻或規則，比較整批結果，再選符合現場目標的一個。 | 第 4 節的工廠每 1,000 件最多可複檢 30 件。在兩個候選門檻中，0.5 會送 28 件，符合容量；0.3 則會送 49 件。 |

**整批要看什麼，也由任務決定。** 管理複檢量時直接數警報件數。有標註答案且重視漏判時，可以看 Recall 或漏判件數；在意送去複檢有多少是真的，可看 Precision；希望同時衡量這兩項，可看 F1。若任務主要看整體答對比例，就看 Accuracy。能估計漏判、誤報各自的代價時，也可以直接比較總成本。

[DLI 04b 手語 Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/blob/main/course_content/tutorials/04b_asl_predictions.ipynb) · [DLI 05a 狗門 Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/blob/main/course_content/tutorials/05a_doggy_door.ipynb)

## 從圖開始

![候選模型的混淆矩陣，Precision 讀預測缺陷欄，Recall 讀實際缺陷列](https://raw.githubusercontent.com/Samuel-ncu/AI-Stastic/main/docs/figures/07-confusion-matrix.svg)

沿欄加總預測缺陷的人數，再沿列加總實際缺陷的人數。用同一個真陽性數計算 Precision 和 Recall。 [閱讀本課講義](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/07-classification-evaluation.md)

## 學習目標與先備知識

先讀第二課的條件機率。學完能自行列出混淆矩陣、手算指標，並依漏判及誤報成本選擇要關注的結果。



## 1. 先定義正類，再計算

**來源摘要 [S6](https://scikit-learn.org/stable/modules/model_evaluation.html)：**Precision 與 Recall 分別以「預測正類」和「實際正類」作為分母；F1 以調和平均彙整兩者。多類別的平均方式會改變整體指標的意義。

正類不代表「好的事」。在本教材自編檢測案例中，1=缺陷，0=良品。先寫清楚這件事，才能知道什麼是漏判。

| 實際＼預測 | 預測良品 0 | 預測缺陷 1 |
|---|---|---|
| 實際良品 0 | TN：正確放行 | FP：誤報、可能多做檢查 |
| 實際缺陷 1 | FN：漏判、可能流出 | TP：正確攔下 |

本課矩陣採列為真實、欄為預測。閱讀不同網站圖表時先看軸標籤，不能只靠位置記憶 TP。



## 2. 用同一張表逐步手算

本教材自編 1,000 件產品，990 良品、10 缺陷。候選模型 TP=8、FN=2、FP=20、TN=970。

$$Accuracy=\frac{TP+TN}{N}=\frac{978}{1000}=97.8\%$$

$$Precision=\frac{TP}{TP+FP}=\frac8{28}\approx28.57\%$$

「系統叫你檢查的 28 件，有多少真的缺陷？」這是 Precision。

$$Recall=\frac{TP}{TP+FN}=\frac8{10}=80\%$$

「真正 10 件缺陷，有多少被抓到？」這是 Recall。

$$F1=\frac{2TP}{2TP+FP+FN}=\frac{16}{38}\approx42.11\%$$

F1 不是 Precision 與 Recall 的普通算術平均，也不直接包含 TN。



## 3. 為什麼 99% 可能比較沒用？

永遠猜良品的基準能得到 990/1000=99% Accuracy，但 Recall=0。它漏掉全部缺陷。本例候選模型 Accuracy 較低，卻抓到 8 件缺陷。

若假設每個 FN 成本 100、FP 成本 1，基準成本為 10×100=1,000；候選成本為 2×100+20=220。這些成本是教學假設，真實專案需向利害關係人確認。



## 4. 閾值如何連到決策？

模型輸出分數後，要指定何時判為正類。降低閾值通常會標出更多正類，同時也可能增加誤報。應在驗證資料比較，而不是看測試資料挑出最好看的閾值。

不同任務也不一定以 F1 為最佳目標。例如若漏判有硬性上限，可先要求 Recall 達標，再比較誤報數或成本。



## 5. 延伸到 DLI 多類別手語

逐類查看誰容易和誰混淆。Macro-F1 對每個類別的 F1 等權平均；Weighted-F1 依各類實際樣本數加權，所以多數類仍可能影響較大。不要只報一個平均而省略類別表現。

如果模型完全不預測正類，Precision 的原分母為 0。程式設 zero_division=0 是回報規則，不表示數學上原本就能做 0/0。

## 6. 從分類延伸到語音與檢索

前五節的指標以「一件產品或一張影像、一次類別判定」為單位。換成語音轉錄或 RAG 時，要先說清楚**要量的是整段文字的編輯錯誤、檢索排序，還是最後的回答**。

| 任務 | 指標與分母 | 例子 |
|---|---|---|
| 語音轉錄 | **WER** 把替換、刪除、插入的詞數加起來，除以參考詞數 | 參考有 4 詞，發生 1 次替換和 1 次刪除，WER 為 $2/4=50\%$ |
| 檢索相關文件 | **Recall@k** 用前 $k$ 名命中的相關文件數，除以全部標註相關文件數 | 總共 2 份相關文件，前 2 名找到 1 份，所以 Recall@2 是 $1/2=0.5$ |
| 檢索排序 | **MRR** 先算第一個相關結果名次的倒數，再對查詢取平均 | 兩題的首個命中名次分別是第 2 和第 1，MRR 為 $(1/2+1)/2=0.75$ |

第 2 節的 Recall 和 Recall@k 都問「真正要找的對象，有多少被找回」，前者以真缺陷為分母，後者以標註相關文件為分母。WER 則允許一段文字累積多次錯誤。接著在 E2 練習按群組報 WER，在 E3 看排名，在 E4 把檢索命中與回答正確分開量。[E2 語音評估](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e2-weak-supervision-wer.md) · [E3 檢索排序](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e3-retrieval-probability.md) · [E4 RAG 消融](https://github.com/Samuel-ncu/AI-Stastic/blob/main/docs/units/e4-rag-ablation.md)

## 自我檢核

1. TP=6、FP=4、FN=2 時，Precision 與 Recall 各是多少？
2. 多數類占 99%，永遠猜多數類是否證明模型學會有用規律？
3. F1 高是否保證誤判總成本最低？

### 參考解答

1. Precision=6/10=60%；Recall=6/8=75%。
2. 沒有；應與其他基準及每類結果比較。
3. 不保證。F1 沒有直接表達所有成本，也不包含 TN。

## 環境

若套件尚未安裝，先執行下一格。安裝需網路，之後的實驗不需下載資料。

In [ ]:
%pip -q install numpy matplotlib scikit-learn

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
y = np.r_[np.zeros(990, dtype=int), np.ones(10, dtype=int)]
baseline = np.zeros_like(y)
candidate = baseline.copy()
candidate[:20] = 1
candidate[990:998] = 1
for name, pred in [('baseline', baseline), ('candidate', candidate)]:
    print(name)
    print('Matrix rows=true, columns=predicted:', confusion_matrix(y, pred, labels=[0, 1]))
    print('Accuracy:', accuracy_score(y, pred),
          'Precision:', precision_score(y, pred, zero_division=0),
          'Recall:', recall_score(y, pred, zero_division=0),
          'F1:', f1_score(y, pred, zero_division=0))
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    print('Cost if FN=100 and FP=1:', 100*fn + fp)
assert accuracy_score(y, baseline) == .99

## 三種任務指標的分母核對

先讀語音的編輯數與檢索的排名，再用分子、分母及答案檢查。完整對齊和排序實驗見 E2、E3。

In [ ]:
from math import isclose

replacements, deletions, insertions, reference_words = 1, 1, 0, 4
wer = (replacements + deletions + insertions) / reference_words
retrieved = ["D1", "D2", "D3"]
relevant = {"D2", "D3"}
recall_at_2 = len(set(retrieved[:2]) & relevant) / len(relevant)
mrr = (1 / 2 + 1 / 1) / 2
assert all(isclose(actual, expected) for actual, expected in zip(
    (wer, recall_at_2, mrr), (0.5, 0.5, 0.75)
))
print(f"WER={wer:.0%}, Recall@2={recall_at_2:.0%}, MRR={mrr:.2f}")

## 如何解讀

基準有 99% Accuracy，卻完全漏掉正類。範例候選模型雖然 Accuracy 較低，但在本例設定的漏判成本下較有價值。零分母指標以 0 回報是明確約定，不代表機率估計。

## 練習

更改 FN 與 FP 成本，找出兩個模型成本相同的條件。

提交修改後的程式、結果圖表及 200 字解讀，列出至少一項限制。若用 AI 協助，說明你如何驗證結果。

## 連結原課程

回到 [DLI Notebook](https://github.com/NVDLI/fundamentals-of-deep-learning-zh/tree/main/course_content/tutorials)，找出本概念對應的資料處理、模型訓練或評估步驟。

## 出處與閱讀建議

- **S6**：[scikit-learn，Metrics and scoring（Log loss、Precision、Recall）](https://scikit-learn.org/stable/modules/model_evaluation.html)。

本課以所列來源的概念作繁體中文摘要，並加入自行編寫的情境、手算、程式及練習；不是原網站的逐段翻譯。標為「來源摘要」者為改寫，標為「自編」者為本教材教學設計；短引文另以引號及「中譯」標明。來源連結可供對照原文，查閱日期：2026-11-17。
